## Starting the BERT model fine tune for the dataset

In [10]:
pip install transformers datasets


[notice] A new release of pip is available: 24.1.1 -> 24.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [120]:
# Ensure the column name is correct
assert 'Comment' in data.columns, "Column 'Comment' not found in the dataset."

In [121]:
data.head()

,Comment ID,Comment,Business Sentiment,Content Sentiment
0,1,"මේ නිෂ්පාදනය නම් ලස්සනයි, නමුත් පාවිච්චිය ටිකක...",Positive,Negative
1,1,rashmika වගේ,Neutral,Positive
2,1,rashmika වෙගේ අඩේ,Neutral,Positive
3,1,සාරංග,Neutral,Neutral
4,1,ඉස්සරහට නිලියක් ශුවර්,Neutral,Positive


### load the SinhalaBERTo and created preproccess funtion 

In [1]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")

In [ ]:
tokenizer

### Analyse the tokenizer

In [ ]:
# Vocabulary size
vocab_size = tokenizer.vocab_size
print(f"Vocabulary Size: {vocab_size}")

In [ ]:
# Tokenization example
text = "rashmika වගේ"
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

In [ ]:
# Token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

In [ ]:
# Decoding token IDs back to text
decoded_text = tokenizer.decode(token_ids)
print(f"Decoded Text: {decoded_text}")

In [ ]:
# Special tokens
special_tokens = tokenizer.special_tokens_map
print(f"Special Tokens: {special_tokens}")

In [ ]:
# Padding and truncation
encoded_input = tokenizer(text, padding='max_length', truncation=True, max_length=10)
print(f"Encoded Input with Padding and Truncation: {encoded_input}")

#### [0, 86, 2698, 81, 4920, 415, 277, 2, 1, 1]
0 is typically the ID for the [CLS] token, which is added at the beginning of the input sequence to signify the start of the sequence.
2 is typically the ID for the [SEP] token, which is added at the end of the input sequence to signify the end of the sequence.
1, 1 are padding tokens added to make the total length 10. These are often represented by the ID 1.

#### [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
This mask is used to distinguish between real tokens and padding tokens.
1 indicates that the corresponding token in input_ids is a real token.
0 indicates that the corresponding token in input_ids is a padding token.  

In [ ]:
# Encoding example
encoded = tokenizer.encode(text)
print(f"Encoded Text: {encoded}")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo")
tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")

### Define a function to predict sentiment

In [ ]:
pip install transformers datasets torch

 The column __index_level_0__ was likely added when you converted the pandas DataFrame to Hugging Face datasets using the Dataset.from_pandas method. 

the train_dataset and test_dataset are not pandas dataframes. They are instances of the datasets.Dataset class from the Hugging Face datasets library. This class is specifically designed for handling and processing large datasets used in machine learning and natural language processing tasks.

 identify all the unique sentiment values in both the "Business Sentiment" and "Content Sentiment" columns.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Specify the file path
file_path = "../../data/mnt/sentiment_Aspect_added.xlsx"

# Load the Excel file into a DataFrame
data = pd.read_excel(file_path)

data.head(10)

,Comment ID,Comment,Business Sentiment,Content Sentiment
0,1,"මේ නිෂ්පාදනය නම් ලස්සනයි, නමුත් පාවිච්චිය ටිකක...",Positive,Negative
1,1,rashmika වගේ,Neutral,Positive
2,1,rashmika වෙගේ අඩේ,Neutral,Positive
3,1,සාරංග,Neutral,Neutral
4,1,ඉස්සරහට නිලියක් ශුවර්,Neutral,Positive
5,1,rashmika මතක් උනා. එයා වගේ ටිකක්,Neutral,Positive
6,1,ව්‍යාකූල උනා පිපිරීම මීට සාරන්ග,Negative,Negative
7,1,මේ අර සිකුරු ඇවිත් ට්‍රාමා එකේ අන්ජලී නේද?,Neutral,Neutral
8,1,එයා නම් ලස්සනයි,Neutral,Positive
9,1,She is ලස්සන,Positive,Positive


In [5]:
# Finding unique values in each column
unique_business_sentiment = data['Business Sentiment'].unique()
# Print unique values
print("Unique values in Business Sentiment:", unique_business_sentiment)


Unique values in Business Sentiment: ['Positive' 'Neutral' 'Negative' nan '...' 'neg']


In [6]:
unique_content_sentiment = data['Content Sentiment'].unique()
print("Unique values in Content Sentiment:", unique_content_sentiment)

Unique values in Content Sentiment: ['Negative' 'Positive' 'Neutral' nan 'Negative ' '...']


In [7]:
# Define replacements
replacement_dict2 = {
    'positive': 'Positive',
    'negative': 'Negative',
    'Negative ':'Negative',
    'Netural': 'Neutral',
    'nan': None,  # Replace 'nan' with None
   '...':None
}

# Replace values in 'Business Sentiment' column
data['Content Sentiment'].replace(replacement_dict2, inplace=True)

# Filter out rows with None (to remove 'nan' and '...')
data = data[data['Content Sentiment'].notna()]

# Check unique values after replacement and filtering
unique_content_sentiment = data['Content Sentiment'].unique()

print("Unique values in Content Sentiment after replacement and filtering:")
print(unique_content_sentiment)

Unique values in Content Sentiment after replacement and filtering:
['Negative' 'Positive' 'Neutral']


In [8]:
# Define replacements
replacement_dict = {
    'positive': 'Positive',
    'eneg': 'Negative',
    'neg': 'Negative',
    'nan': None,  # Replace 'nan' with None
    '...': None    # Replace '...' with None
}

# Replace values in 'Business Sentiment' column
data['Business Sentiment'].replace(replacement_dict, inplace=True)

# Filter out rows with None (to remove 'nan' and '...')
data = data[data['Business Sentiment'].notna()]

# Check unique values after replacement and filtering
unique_business_sentiment = data['Business Sentiment'].unique()

print("Unique values in Business Sentiment after replacement and filtering:")
print(unique_business_sentiment)

Unique values in Business Sentiment after replacement and filtering:
['Positive' 'Neutral' 'Negative']


In [9]:
# Finding unique values in each column
unique_business_sentiment = data['Business Sentiment'].unique()
unique_content_sentiment = data['Content Sentiment'].unique()

# Print unique values
print("Unique values in Business Sentiment:", unique_business_sentiment)
print("Unique values in Content Sentiment:", unique_content_sentiment)

Unique values in Business Sentiment: ['Positive' 'Neutral' 'Negative']
Unique values in Content Sentiment: ['Negative' 'Positive' 'Neutral']


In [10]:
data.head()

,Comment ID,Comment,Business Sentiment,Content Sentiment
0,1,"මේ නිෂ්පාදනය නම් ලස්සනයි, නමුත් පාවිච්චිය ටිකක...",Positive,Negative
1,1,rashmika වගේ,Neutral,Positive
2,1,rashmika වෙගේ අඩේ,Neutral,Positive
3,1,සාරංග,Neutral,Neutral
4,1,ඉස්සරහට නිලියක් ශුවර්,Neutral,Positive


In [11]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [12]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform 'Business Sentiment' and 'Content Sentiment' columns
data['Business Sentiment'] = label_encoder.fit_transform(data['Business Sentiment'])
data['Content Sentiment'] = label_encoder.fit_transform(data['Content Sentiment'])

In [13]:
# Find unique values in 'Business Sentiment' and 'Content Sentiment' columns
unique_business_sentiments = data['Business Sentiment'].unique()
unique_content_sentiments = data['Content Sentiment'].unique()

# Print the unique values
print("Unique Business Sentiments:", unique_business_sentiments)
print("Unique Content Sentiments:", unique_content_sentiments)

Unique Business Sentiments: [2 1 0]
Unique Content Sentiments: [0 2 1]


In [14]:
data.shape

(10927, 4)

In [15]:
data.duplicated().sum()

1867

In [16]:
data = data.dropna()

In [17]:
data.isnull().sum()

Comment ID            0
Comment               0
Business Sentiment    0
Content Sentiment     0
dtype: int64

In [18]:
data.head(10)

,Comment ID,Comment,Business Sentiment,Content Sentiment
0,1,"මේ නිෂ්පාදනය නම් ලස්සනයි, නමුත් පාවිච්චිය ටිකක...",2,0
1,1,rashmika වගේ,1,2
2,1,rashmika වෙගේ අඩේ,1,2
3,1,සාරංග,1,1
4,1,ඉස්සරහට නිලියක් ශුවර්,1,2
5,1,rashmika මතක් උනා. එයා වගේ ටිකක්,1,2
6,1,ව්‍යාකූල උනා පිපිරීම මීට සාරන්ග,0,0
7,1,මේ අර සිකුරු ඇවිත් ට්‍රාමා එකේ අන්ජලී නේද?,1,1
8,1,එයා නම් ලස්සනයි,1,2
9,1,She is ලස්සන,2,2


In [19]:
import re
def remove_links(text):
    if isinstance(text, str):
        # Remove URLs starting with http or https
        text = re.sub(r'http\S+', '', text)
        # Remove URLs starting with www
        text = re.sub(r'www\S+', '', text)
        return text
        
# Apply the function to the 'Comment' column
data['Comment'] = data['Comment'].apply(remove_links).astype(str)

import string
def remove_punctuation(text):
    if isinstance(text, str):
        # Remove punctuation marks
        text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# Apply the function to the 'Comment' column
data['Comment'] = data['Comment'].apply(remove_punctuation)

data["Comment"] = data['Comment'].str.replace('\d+', '', regex=True)

In [20]:
data.head()

,Comment ID,Comment,Business Sentiment,Content Sentiment
0,1,මේ නිෂ්පාදනය නම් ලස්සනයි නමුත් පාවිච්චිය ටිකක්...,2,0
1,1,rashmika වගේ,1,2
2,1,rashmika වෙගේ අඩේ,1,2
3,1,සාරංග,1,1
4,1,ඉස්සරහට නිලියක් ශුවර්,1,2


In [21]:
# Function to check if a comment contains only Sinhala text
def is_sinhala_text(comment):
    # Regular expression to match only Sinhala characters and spaces
    sinhala_pattern = re.compile(r'^[\u0D80-\u0DFF\s]+$')
    return bool(sinhala_pattern.match(comment))
    
data['Comment'] = data['Comment'].astype(str)
# Filter rows with only Sinhala text and remove the 'Comment ID' column
filtered_data = data[data['Comment'].apply(is_sinhala_text)].drop(columns=['Comment ID'])

In [22]:
filtered_data.head()

,Comment,Business Sentiment,Content Sentiment
0,මේ නිෂ්පාදනය නම් ලස්සනයි නමුත් පාවිච්චිය ටිකක්...,2,0
3,සාරංග,1,1
4,ඉස්සරහට නිලියක් ශුවර්,1,2
8,එයා නම් ලස්සනයි,1,2
10,සේවාව මට අකමැති නමුත් කණ්ඩායම හොඳයි,0,2


In [23]:
# Find unique values in 'Business Sentiment' and 'Content Sentiment' columns
unique_business_sentiments = filtered_data['Business Sentiment'].unique()
unique_content_sentiments = filtered_data['Content Sentiment'].unique()

# Print the unique values
print("Unique Business Sentiments:", unique_business_sentiments)
print("Unique Content Sentiments:", unique_content_sentiments)

Unique Business Sentiments: [2 1 0]
Unique Content Sentiments: [0 1 2]


In [24]:
filtered_data.head()

,Comment,Business Sentiment,Content Sentiment
0,මේ නිෂ්පාදනය නම් ලස්සනයි නමුත් පාවිච්චිය ටිකක්...,2,0
3,සාරංග,1,1
4,ඉස්සරහට නිලියක් ශුවර්,1,2
8,එයා නම් ලස්සනයි,1,2
10,සේවාව මට අකමැති නමුත් කණ්ඩායම හොඳයි,0,2


In [25]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [26]:
from sklearn.model_selection import train_test_split
# Splitting data into train and eval sets
train_data, eval_data = train_test_split(filtered_data, test_size=0.2, random_state=42)

#### Define Custom Dataset Class

Prepare the Dataset and DataLoader

In [29]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len # The maximum number of words to use from each comment.

    def __len__(self):
        return len(self.data)   # This function returns how many items are in our dataset.

    def __getitem__(self, index):              # This function gets one item from our dataset.
        comment = str(self.data.loc[index, 'Comment'])
        inputs = self.tokenizer.encode_plus(
            comment,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = inputs['input_ids'].flatten()   #  The actual numbers representing words in the comment.
        attention_mask = inputs['attention_mask'].flatten() # A mask to tell the model which parts are important (the actual words) and which are just padding.
        label = torch.tensor(self.data.loc[index, 'Content Sentiment'], dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': label
        }

tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")
model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo", num_labels=3)  # Adjust num_labels to 3

train_data.reset_index(drop=True, inplace=True)  # Reset the index
eval_data.reset_index(drop=True, inplace=True)  # Reset the index

max_len = 128
train_dataset = SentimentDataset(train_data, tokenizer, max_len)
eval_dataset = SentimentDataset(eval_data, tokenizer, max_len)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)   # helper class in PyTorch that makes it easier to load data in batches

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="steps",     # evaluation strategy to log metrics at each logging step
    eval_steps=10,                   # evaluation steps
    load_best_model_at_end=True,     # load the best model when finished training (for early stopping)
    metric_for_best_model="accuracy",# the metric to use to compare the models
    greater_is_better=True           # whether a higher metric is better
)

def compute_metrics(p):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    preds = p.predictions.argmax(-1)
    accuracy = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted')
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # Added eval_dataset
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at keshan/SinhalaBERTo and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
10,1.127300,1.100385,0.342037,0.405712,0.342037,0.267062
20,1.087200,1.040860,0.419495,0.434766,0.419495,0.398955
30,0.999200,0.979136,0.512620,0.416314,0.512620,0.438223
40,0.939100,0.944695,0.541340,0.446239,0.541340,0.395895
50,0.925600,0.937684,0.539600,0.458344,0.539600,0.381318
60,0.892400,0.928036,0.543081,0.460352,0.543081,0.388257
70,0.920700,0.914479,0.554395,0.456233,0.554395,0.480225
80,0.889900,0.908552,0.550044,0.483916,0.550044,0.514713
90,0.869600,0.895008,0.561358,0.458523,0.561358,0.475673
100,0.867200,0.887488,0.563969,0.469040,0.563969,0.481385


D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier

TrainOutput(global_step=864, training_loss=0.7483966336758049, metrics={'train_runtime': 18501.7288, 'train_samples_per_second': 0.745, 'train_steps_per_second': 0.047, 'total_flos': 456623266249728.0, 'train_loss': 0.7483966336758049, 'epoch': 3.0})

In [ ]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len # The maximum number of words to use from each comment.

    def __len__(self):
        return len(self.data)   # This function returns how many items are in our dataset.

    def __getitem__(self, index):              # This function gets one item from our dataset.
        comment = str(self.data.loc[index, 'Comment'])
        inputs = self.tokenizer.encode_plus(
            comment,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = inputs['input_ids'].flatten()   #  The actual numbers representing words in the comment.
        attention_mask = inputs['attention_mask'].flatten() # A mask to tell the model which parts are important (the actual words) and which are just padding.
        label = torch.tensor(self.data.loc[index, 'Business Sentiment'], dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': label
        }

tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")
model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo", num_labels=3)  # Adjust num_labels to 3

train_data.reset_index(drop=True, inplace=True)  # Reset the index
eval_data.reset_index(drop=True, inplace=True)  # Reset the index

max_len = 128
train_dataset = SentimentDataset(train_data, tokenizer, max_len)
eval_dataset = SentimentDataset(eval_data, tokenizer, max_len)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)   # helper class in PyTorch that makes it easier to load data in batches

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=64,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="steps",     # evaluation strategy to log metrics at each logging step
    eval_steps=10,                   # evaluation steps
    load_best_model_at_end=True,     # load the best model when finished training (for early stopping)
    metric_for_best_model="accuracy",# the metric to use to compare the models
    greater_is_better=True           # whether a higher metric is better
)

def compute_metrics(p):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    preds = p.predictions.argmax(-1)
    accuracy = accuracy_score(p.label_ids, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted')
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # Added eval_dataset
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at keshan/SinhalaBERTo and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\transformers\training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
10,1.089000,1.051711,0.544822,0.482600,0.544822,0.507273
20,1.019200,0.970208,0.651871,0.465737,0.651871,0.522452
30,0.896700,0.882232,0.657093,0.432147,0.657093,0.521393
40,0.841700,0.835285,0.657093,0.431771,0.657093,0.521119
50,0.814700,0.820315,0.657093,0.431771,0.657093,0.521119
60,0.839200,0.811138,0.688425,0.662056,0.688425,0.590831
70,0.813800,0.804518,0.697128,0.646917,0.697128,0.614678
80,0.764900,0.787872,0.693647,0.623515,0.693647,0.619277
90,0.792000,0.786724,0.691906,0.609941,0.691906,0.635884
100,0.771000,0.773783,0.698869,0.625744,0.698869,0.630824


D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\UOM\L4S1\Research\Implementation\FYP-Research\env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier

In [70]:
# Define the compute_metrics function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # Assuming binary classification for simplicity
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

In [77]:
# from transformers import default_data_collator

# Initialize the Trainer for Content Sentiment
trainer_content = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=train_dataset.map(preprocess_content, batched=True),   # training dataset with preprocessing
    eval_dataset=test_dataset.map(preprocess_content, batched=True),      # evaluation dataset with preprocessing
    compute_metrics=compute_metrics,     # evaluation metrics
)

# Fine-tune the model for Business Sentiment
trainer_content.train()

Map:   0%|          | 0/567 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.544717,0.739437
2,No log,0.494225,0.732394
3,No log,0.459979,0.774648


TrainOutput(global_step=108, training_loss=0.47586423379403575, metrics={'train_runtime': 642.0593, 'train_samples_per_second': 2.649, 'train_steps_per_second': 0.168, 'total_flos': 56331761278464.0, 'train_loss': 0.47586423379403575, 'epoch': 3.0})

In [79]:
# Initialize the Trainer for Business Sentiment
trainer_business = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=train_dataset.map(preprocess_business, batched=True),  # training dataset with preprocessing
    eval_dataset=test_dataset.map(preprocess_business, batched=True),     # evaluation dataset with preprocessing
    compute_metrics=compute_metrics,     # evaluation metrics
)

# Fine-tune the model for Business Sentiment
trainer_business.train()

Map:   0%|          | 0/567 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.506763,0.781690
2,No log,0.530166,0.753521
3,No log,0.554204,0.802817


TrainOutput(global_step=108, training_loss=0.17344352934095594, metrics={'train_runtime': 647.9758, 'train_samples_per_second': 2.625, 'train_steps_per_second': 0.167, 'total_flos': 56331761278464.0, 'train_loss': 0.17344352934095594, 'epoch': 3.0})

In [81]:
# Save the model
trainer_content.save_model("./results_content")
trainer_business.save_model("./results_business")

##### Switch case model running

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Load the dataset
file_path = '../../data/mnt/sentiment_Aspect_added.xlsx'
data = pd.read_excel(file_path)

# Ensure all comments are strings
data['Comment'] = data['Comment'].astype(str)

# Split the dataset into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

# Drop the __index_level_0__ column
train_dataset = train_dataset.remove_columns(['__index_level_0__'])
test_dataset = test_dataset.remove_columns(['__index_level_0__'])

# Verify the split
print(f"Training set size: {len(train_dataset)}")
print(f"Testing set size: {len(test_dataset)}")

Training set size: 568
Testing set size: 142


In [3]:
# Drop rows with NaN values in 'Business Sentiment' and 'Content Sentiment'
data = data.dropna(subset=['Business Sentiment', 'Content Sentiment'])

# Verify that only the expected sentiments are present
unique_business_sentiments = data['Business Sentiment'].unique()
unique_content_sentiments = data['Content Sentiment'].unique()

print("Unique Business Sentiments:", unique_business_sentiments)
print("Unique Content Sentiments:", unique_content_sentiments)

Unique Business Sentiments: ['Positive' 'Neutral' 'Negative']
Unique Content Sentiments: ['Positive' 'Neutral' 'Negative' 'Negative (Sarcastic criticism)'
 'Negative (could be sarcastic)' 'Negative (Could be sarcastic)'
 'Negative (Comparison)']


In [4]:
# Replace specific "Negative" related sentiments with "Negative"
negative_related_sentiments = ['Negative (Sarcastic criticism)', 
                               'Negative (could be sarcastic)', 
                               'Negative (Could be sarcastic)', 
                               'Negative (Comparison)',
                              ]

data['Content Sentiment'] = data['Content Sentiment'].replace(negative_related_sentiments, 'Negative')
data['Business Sentiment'] = data['Business Sentiment'].replace(negative_related_sentiments, 'Negative')

# Drop rows with NaN values in 'Business Sentiment' and 'Content Sentiment'
data = data.dropna(subset=['Business Sentiment', 'Content Sentiment'])

# Verify that only the expected sentiments are present
unique_business_sentiments = data['Business Sentiment'].unique()
unique_content_sentiments = data['Content Sentiment'].unique()

print("Unique Business Sentiments:", unique_business_sentiments)
print("Unique Content Sentiments:", unique_content_sentiments)

Unique Business Sentiments: ['Positive' 'Neutral' 'Negative']
Unique Content Sentiments: ['Positive' 'Neutral' 'Negative']


In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo")
tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at keshan/SinhalaBERTo and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
import torch

def predict_sentiment(comment):
    inputs = tokenizer(comment, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    return predicted_class_id

positive_sentences = ["ඒක හොඳයි.", "මට හරි සතුටුයි.", "ශුවර්", "විශ්වාසයි","මාරයි"]
negative_sentences = ["ඒක නරකයි.", "මට සතුටු නෑ", "කැතයි"]

# positive_sentences = ["ශුවර්"]
# negative_sentences = ["ඒක සුපිරි නෑ.", "ලස්සනයි ඒක"]


for sentence in positive_sentences:
    print(f"Positive sentence: '{sentence}' -> Label: {predict_sentiment(sentence)}")
for sentence in negative_sentences:
    print(f"Negative sentence: '{sentence}' -> Label: {predict_sentiment(sentence)}")

Positive sentence: 'ඒක හොඳයි.' -> Label: 1
Positive sentence: 'මට හරි සතුටුයි.' -> Label: 1
Positive sentence: 'ශුවර්' -> Label: 0
Positive sentence: 'විශ්වාසයි' -> Label: 1
Positive sentence: 'මාරයි' -> Label: 0
Negative sentence: 'ඒක නරකයි.' -> Label: 1
Negative sentence: 'මට සතුටු නෑ' -> Label: 0
Negative sentence: 'කැතයි' -> Label: 0


In [36]:
# Sample comments
comments = [
    "rashmika වගේ",  # Positive
    "සාරංග",        # Neutral
    "ගැජට් එක නරකයි", # Negative
    "ඒක හොඳයි.", "මට හරි සතුටුයි.", "ශුවර්", "විශ්වාසයි","කැතයි","එල"
]

# Tokenize the comments
tokens = tokenizer(comments, padding=True, truncation=True, return_tensors='pt')

In [37]:
# Get model predictions
outputs = model(**tokens)
logits = outputs.logits
predictions = torch.argmax(logits, dim=1).tolist()

In [38]:
print("Predictions:", predictions)

Predictions: [0, 0, 0, 1, 1, 0, 1, 0, 0]


In [39]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [41]:
import pandas as pd

# Load your dataset
df = pd.read_excel('../../data/mnt/sentiment_Aspect_added.xlsx')

# Mapping sentiments to numerical labels
sentiment_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['business_label'] = df['Business Sentiment'].map(sentiment_map)
df['content_label'] = df['Content Sentiment'].map(sentiment_map)

In [42]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("keshan/SinhalaBERTo")

# Load the model and adjust for three labels
model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo", num_labels=3)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at keshan/SinhalaBERTo and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [44]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset

# Ensure comments are passed as a list of strings
comments = df['Comment'].astype(str).tolist()

# Tokenize the comments
tokens = tokenizer(comments, padding=True, truncation=True, return_tensors='pt')

def create_dataloaders(labels, tokens, batch_size=16):
    train_tokens, val_tokens, train_labels, val_labels = train_test_split(
        tokens['input_ids'], labels, test_size=0.2, random_state=42)
    
    train_dataset = TensorDataset(train_tokens, torch.tensor(train_labels))
    val_dataset = TensorDataset(val_tokens, torch.tensor(val_labels))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    return train_loader, val_loader

# Create dataloaders for business sentiment
business_train_loader, business_val_loader = create_dataloaders(df['business_label'].values, tokens)

# Create dataloaders for content sentiment
content_train_loader, content_val_loader = create_dataloaders(df['content_label'].values, tokens)


In [46]:
from transformers import AdamW
from tqdm import tqdm

In [48]:
# Function to set up the model, optimizer, and loss function
def setup_model():
    model = AutoModelForSequenceClassification.from_pretrained("keshan/SinhalaBERTo", num_labels=3)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    loss_fn = torch.nn.CrossEntropyLoss()
    return model, optimizer, loss_fn

# Training loop
def train_model(model, optimizer, loss_fn, train_loader, epochs=3):
    model.train()
    for epoch in range(epochs):
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            input_ids = batch[0]
            labels = batch[1]
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}, Loss: {loss.item()}")